<a href="https://colab.research.google.com/github/mohamed2ayman/Claude/blob/main/ModelBasedGrading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

#Load new Variable
from dotenv import load_dotenv
load_dotenv()

# Create API Client
import os
from anthropic import Anthropic

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

client = Anthropic()
model = "claude-sonnet-4-5"

In [27]:
# Add user message, while will tak list of messages and text. It's reason is to append messages together and save it to history.

def add_user_message(messages, text):
  user_message = {"role":"user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)


def chat(messages, system=None, temperature =1.0, stop_sequences=None ):
  params = {
      "model": model,
      "max_tokens": 1000,
      "messages": messages,
      "temperature": temperature
  }

  if system:
    params["system"] = system

  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  message = client.messages.create(**params)

  return message.content[0].text

In [28]:
# Creating an Evaluation Dataset for Prompt Evaluation

import json

def generate_dataset():
  prompt= """

Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python,
JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python,
JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "'''json")
  text = chat(messages, stop_sequences=["'''"])
  return json.loads(text)


In [29]:
# Dumping the dataset evaluation in a JSON file
import json
dataset= generate_dataset()

with open("dataset.json", "w") as f:
  json.dump(dataset, f, indent=2)

In [30]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt= f"""

Please solve the following task:

{test_case["task"]}

"""

  messages=[]
  add_user_message(messages, prompt)
  output = chat(messages)
  return output


In [46]:
def grade_by_model(test_case, output):
  # Create evaluation prompt
    eval_prompt =f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Original Task:

    <task>
    Task: {test_case["task"]}
    </task>

    Solution to evaluate:

    <solution>
    Solution: {output}
    </solution>

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10


    Respond with JSON. Keep your response concise and direct.
    Example response shape:

   {{

      "strengths": string[],
      "weaknesses": string[],
      "reasoning": string,
      "score": number
   }}

      """



    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [47]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the resullt"""
  output = run_prompt(test_case)

  # TODO -Grading
  model_grade = grade_by_model(test_case, output)
  score = model_grade["score"]
  reasoning = model_grade["reasoning"]

  return {

          "output": output,
          "test_case": test_case,
          "score": score,
          "reasoning": reasoning
  }


In [48]:

from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls run_test_case for each test case"""
  results = []
  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result["score"] for result in results])
  print(f"Average score: {average_score}")

  return results

In [49]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

Average score: 7


In [50]:
print(json.dumps(results, indent=2))

[
  {
    "output": "I'll help you write a Python function to extract the service name from an AWS ARN string.\n\n```python\ndef extract_service_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the service name from an AWS ARN string.\n    \n    AWS ARN format: arn:partition:service:region:account-id:resource-id\n    \n    Args:\n        arn: AWS ARN string\n        \n    Returns:\n        The service name extracted from the ARN\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n        \n    Examples:\n        >>> extract_service_from_arn('arn:aws:s3:::my-bucket')\n        's3'\n        >>> extract_service_from_arn('arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0')\n        'ec2'\n        >>> extract_service_from_arn('arn:aws:iam::123456789012:user/JohnDoe')\n        'iam'\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    parts = arn.split(':')\n    \n    # ARN mu